In [ ]:
import csv
import random
import uuid
from datetime import datetime, timezone

output_path = "/Volumes/workspace/gagealspach/ams_demo_source/batch_source"

service_levels = ["STANDARD", "EXPEDITED", "PRIORITY"]
equipment_types = ["DRY_VAN", "REEFER", "FLATBED"]
terminals = ["Reno", "Sparks", "Sacramento", "Fernley", "Carson City"]

now = datetime.now(timezone.utc)

# Pull current shipment IDs from the streaming Silver table
streaming_shipments = (
    spark.table("workspace.gagealspach.shipments_stream_silver")
    .select("shipment_id")
    .distinct()
    .limit(25)
    .collect()
)

if not streaming_shipments:
    raise Exception("No shipments currently exist in shipments_stream_silver")

rows = []

for row in streaming_shipments:
    rows.append({
        "shipment_id": row["shipment_id"],
        "service_level": random.choice(service_levels),
        "equipment_type": random.choice(equipment_types),
        "origin_terminal": random.choice(terminals),
        "destination_terminal": random.choice(terminals),
        "weight_lbs": random.randint(500, 45000),
        "batch_created_ts": now.strftime("%Y-%m-%dT%H:%M:%S"),
    })

filename = f"batch_{now:%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:6]}.csv"
filepath = f"{output_path}/{filename}"

with open(filepath, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {filename} with {len(rows)} shipment enrichments")